In [240]:
import pandas as pd
df = pd.read_pickle('final_cleaned.pkl')
df.head()

,insurance_type,label,prev_readmit_group,comorbidities_group,los_group,medications_group,followup_visits_group,risk_score_bin,dc_location,primary_dx_tier,age_bin
0,Medicare,1,1,five or more,6 to 8,8 to 14,5 or more,7,Home Health,lower,4
1,Private,1,2,four,6 to 8,6,4,7,Home Health,lower,4
2,Medicare,1,2,five or more,6 to 8,8 to 14,5 or more,7,Skilled Nursing,lower,5
3,Medicare,1,2,five or more,9+,8 to 14,4,7,Skilled Nursing,higher,5
4,Uninsured,1,1,four,9+,6,2,6,Home Health,higher,4


In [241]:
feature_cols = [
    'insurance_type',
    'prev_readmit_group',
    'comorbidities_group',
    'los_group',
    'medications_group',
    'followup_visits_group',
    'risk_score_bin',
    'dc_location',
    'primary_dx_tier',
    'age_bin'
]
df[feature_cols] = df[feature_cols].astype('category')


In [242]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = df[feature_cols]
y = df['label']

X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.816875

In [243]:
import numpy as np

coef = pd.Series(model.coef_[0], index=X_encoded.columns)
coef_abs = coef.abs().sort_values(ascending=False)
coef_abs


risk_score_bin_7                   2.093349
risk_score_bin_6                   1.273727
prev_readmit_group_2               1.036329
age_bin_5                          0.923792
risk_score_bin_5                   0.857017
risk_score_bin_4                   0.679981
comorbidities_group_one            0.578479
prev_readmit_group_1               0.534910
risk_score_bin_3                   0.498974
age_bin_4                          0.494622
primary_dx_tier_lower              0.462004
medications_group_7                0.397419
comorbidities_group_two            0.348276
dc_location_Home/Rehab             0.298017
risk_score_bin_2                   0.240433
medications_group_6                0.199807
comorbidities_group_three          0.199074
age_bin_2                          0.198936
los_group_6 to 8                   0.157386
medications_group_5                0.155543
medications_group_15+              0.120245
insurance_type_Private             0.113460
followup_visits_group_4         

In [244]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)
confusion_matrix(y_test, y_pred)

array([[ 160,  191],
       [ 102, 1147]])

In [245]:
results = {
    "TP": 1151,
    "TN": 166,
    "FP": 185,
    "FN": 98

}
results


{'TP': 1151, 'TN': 166, 'FP': 185, 'FN': 98}

In [246]:
coef = pd.Series(model.coef_[0], index=X_encoded.columns)

# Sort alphabetically by feature name
coef_sorted_by_name = coef.sort_index()

coef_sorted_by_name



age_bin_2                          0.198936
age_bin_3                          0.013103
age_bin_4                          0.494622
age_bin_5                          0.923792
comorbidities_group_four           0.023774
comorbidities_group_one           -0.578479
comorbidities_group_three         -0.199074
comorbidities_group_two           -0.348276
dc_location_Home/Rehab            -0.298017
dc_location_Skilled Nursing       -0.026073
followup_visits_group_2           -0.025867
followup_visits_group_3            0.015503
followup_visits_group_4            0.107532
followup_visits_group_5 or more   -0.100763
insurance_type_Medicare           -0.032039
insurance_type_Private            -0.113460
insurance_type_Uninsured           0.093293
los_group_5                       -0.072156
los_group_6 to 8                  -0.157386
los_group_9+                       0.097274
medications_group_15+              0.120245
medications_group_3                0.000213
medications_group_4             

In [247]:
def show_raw_coefs_for(model, feature):
    coefs = pd.Series(model.coef_.flatten(), index=model.feature_names_in_)
    # return coefs[coefs.index.str.startswith(feature + "_")]
    print('baseline = 0.0000')
    print(coefs[coefs.index.str.startswith(feature + "_")])



In [248]:
bins = df['age_bin'].value_counts()
print(bins)
show_raw_coefs_for(model, "age_bin")

age_bin
4    2864
5    2211
3    1956
2     826
1     143
Name: count, dtype: int64
baseline = 0.0000
age_bin_2    0.198936
age_bin_3    0.013103
age_bin_4    0.494622
age_bin_5    0.923792
dtype: float64


In [249]:
# lumps age bins 1-3 into the same bin
df['age_bin'] = df['age_bin'].map({
    2: 1,
    3: 1,
    4: 2,
    5: 3
})
df['age_bin'].value_counts()

age_bin
2.0    2864
1.0    2782
3.0    2211
Name: count, dtype: int64

In [250]:
df.head()

,insurance_type,label,prev_readmit_group,comorbidities_group,los_group,medications_group,followup_visits_group,risk_score_bin,dc_location,primary_dx_tier,age_bin
0,Medicare,1,1,five or more,6 to 8,8 to 14,5 or more,7,Home Health,lower,2.0
1,Private,1,2,four,6 to 8,6,4,7,Home Health,lower,2.0
2,Medicare,1,2,five or more,6 to 8,8 to 14,5 or more,7,Skilled Nursing,lower,3.0
3,Medicare,1,2,five or more,9+,8 to 14,4,7,Skilled Nursing,higher,3.0
4,Uninsured,1,1,four,9+,6,2,6,Home Health,higher,2.0


In [251]:
bins = df['medications_group'].value_counts()
print(bins)
show_raw_coefs_for(model, "medications_group")

medications_group
8 to 14    4191
6          1767
7           883
4           776
3           195
5           104
2            76
15+           8
Name: count, dtype: int64
baseline = 0.0000
medications_group_3          0.000213
medications_group_4          0.028028
medications_group_5          0.155543
medications_group_6         -0.199807
medications_group_7         -0.397419
medications_group_15+        0.120245
medications_group_8 to 14    0.100728
dtype: float64


In [252]:
df = df.drop(columns=['medications_group'])

In [253]:
df.head()

,insurance_type,label,prev_readmit_group,comorbidities_group,los_group,followup_visits_group,risk_score_bin,dc_location,primary_dx_tier,age_bin
0,Medicare,1,1,five or more,6 to 8,5 or more,7,Home Health,lower,2.0
1,Private,1,2,four,6 to 8,4,7,Home Health,lower,2.0
2,Medicare,1,2,five or more,6 to 8,5 or more,7,Skilled Nursing,lower,3.0
3,Medicare,1,2,five or more,9+,4,7,Skilled Nursing,higher,3.0
4,Uninsured,1,1,four,9+,2,6,Home Health,higher,2.0


In [254]:
group = 'prev_readmit_group'

bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

prev_readmit_group
1    3800
2    3723
0     477
Name: count, dtype: int64
baseline = 0.0000
prev_readmit_group_1    0.534910
prev_readmit_group_2    1.036329
dtype: float64


In [255]:
group = 'comorbidities_group'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)
df = df.drop(columns=['comorbidities_group'])

comorbidities_group
five or more    3437
four            2397
three           1520
two              534
one              112
Name: count, dtype: int64
baseline = 0.0000
comorbidities_group_four     0.023774
comorbidities_group_one     -0.578479
comorbidities_group_three   -0.199074
comorbidities_group_two     -0.348276
dtype: float64


In [256]:
df.head()

,insurance_type,label,prev_readmit_group,los_group,followup_visits_group,risk_score_bin,dc_location,primary_dx_tier,age_bin
0,Medicare,1,1,6 to 8,5 or more,7,Home Health,lower,2.0
1,Private,1,2,6 to 8,4,7,Home Health,lower,2.0
2,Medicare,1,2,6 to 8,5 or more,7,Skilled Nursing,lower,3.0
3,Medicare,1,2,9+,4,7,Skilled Nursing,higher,3.0
4,Uninsured,1,1,9+,2,6,Home Health,higher,2.0


In [257]:
group = 'los_group'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

los_group
6 to 8    4205
9+        2851
5          764
3 to 4     180
Name: count, dtype: int64
baseline = 0.0000
los_group_5        -0.072156
los_group_6 to 8   -0.157386
los_group_9+        0.097274
dtype: float64


In [258]:
group = 'followup_visits_group'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

followup_visits_group
2            2469
4            2307
5 or more    1986
3             992
0 to 1        246
Name: count, dtype: int64
baseline = 0.0000
followup_visits_group_2           -0.025867
followup_visits_group_3            0.015503
followup_visits_group_4            0.107532
followup_visits_group_5 or more   -0.100763
dtype: float64


In [259]:
group = 'risk_score_bin'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

risk_score_bin
7    3945
5    1069
2     832
6     733
3     594
4     491
1     336
Name: count, dtype: int64
baseline = 0.0000
risk_score_bin_2    0.240433
risk_score_bin_3    0.498974
risk_score_bin_4    0.679981
risk_score_bin_5    0.857017
risk_score_bin_6    1.273727
risk_score_bin_7    2.093349
dtype: float64


In [260]:
group = 'dc_location'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

dc_location
Home Health        6065
Skilled Nursing    1568
Home/Rehab          367
Name: count, dtype: int64
baseline = 0.0000
dc_location_Home/Rehab        -0.298017
dc_location_Skilled Nursing   -0.026073
dtype: float64


In [261]:

df['dc_location'] = df['dc_location'].map({
    'Home Health': 'HH/SNF',
    'Skilled Nursing': 'HH/SNF',
    'Home/Rehab': 'Home/Rehab'
})

In [262]:
df['dc_location'].value_counts()

dc_location
HH/SNF        7633
Home/Rehab     367
Name: count, dtype: int64

In [263]:
group = 'primary_dx_tier'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

primary_dx_tier
lower     4397
higher    3603
Name: count, dtype: int64
baseline = 0.0000
primary_dx_tier_lower   -0.462004
dtype: float64


In [264]:
group = 'insurance_type'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

insurance_type
Private      3052
Medicare     2542
Medicaid     1598
Uninsured     808
Name: count, dtype: int64
baseline = 0.0000
insurance_type_Medicare    -0.032039
insurance_type_Private     -0.113460
insurance_type_Uninsured    0.093293
dtype: float64


In [265]:
df['insurance_type'] = df['insurance_type'].map({
    'Private': 'Private',
    'Uninsured': 'Uninsured',
    'Medicare': 'Medicare/Medicaid',
    'Medicaid': 'Medicare/Medicaid'
})

In [266]:
df['insurance_type'].value_counts()

insurance_type
Medicare/Medicaid    4140
Private              3052
Uninsured             808
Name: count, dtype: int64

In [267]:
df.to_pickle('coef_cleaned_one.pkl')